# 🔧 Notebook 02 : Prétraitement & Feature Engineering - Détection de Fraude

**Auteur** : Oumaro Titans DJIGUIMDE  
**Date** : Février 2026  
**Objectif** : Préparer les données pour la modélisation Machine Learning

---

## 🎯 Objectifs de ce notebook

1. Encoder les variables catégorielles
2. Normaliser les variables numériques
3. Créer de nouvelles features (Feature Engineering)
4. Gérer le déséquilibre des classes
5. Diviser les données (Train/Test)
6. Sauvegarder les données préparées

## 📦 Importation des bibliothèques

In [ ]:
# Manipulation de données
import pandas as pd
import numpy as np

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Utilitaires
import warnings
warnings.filterwarnings('ignore')

# Configuration
pd.set_option('display.max_columns', None)
np.random.seed(42)

print("✅ Bibliothèques importées avec succès")

## 1️⃣ Chargement des données

In [ ]:
# Chargement
df = pd.read_csv('../data/transactions.csv', parse_dates=['date'])

print("="*70)
print("📊 DATASET CHARGÉ")
print("="*70)
print(f"Transactions : {len(df):,}")
print(f"Features : {len(df.columns)}")
print(f"Fraudes : {df['is_fraud'].sum():,} ({df['is_fraud'].mean()*100:.2f}%)")
print("="*70)

In [ ]:
# Aperçu
df.head()

## 2️⃣ Feature Engineering

Création de nouvelles variables à partir des existantes

In [ ]:
print("➕ CRÉATION DE NOUVELLES FEATURES")
print("="*70)

# 1. Features temporelles
df['day_of_week'] = df['date'].dt.dayofweek  # 0=Lundi, 6=Dimanche
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day

print("✅ Features temporelles créées :")
print("   • day_of_week, is_weekend, month, day")

# 2. Catégorisation de l'heure
def categorize_hour(hour):
    if 0 <= hour < 6:
        return 'nuit'  # Très risqué
    elif 6 <= hour < 12:
        return 'matin'
    elif 12 <= hour < 18:
        return 'apres_midi'
    else:
        return 'soir'  # Risqué

df['period_of_day'] = df['hour'].apply(categorize_hour)
print("\n✅ Période du jour créée (nuit, matin, après-midi, soir)")

# 3. Catégorisation du montant
df['amount_category'] = pd.cut(df['amount'], 
                                bins=[0, 10000, 50000, 100000, float('inf')],
                                labels=['petit', 'moyen', 'eleve', 'tres_eleve'])
print("\n✅ Catégorie de montant créée (petit, moyen, élevé, très élevé)")

# 4. Log du montant (normalisation)
df['amount_log'] = np.log1p(df['amount'])  # log(1+x) pour éviter log(0)
print("\n✅ Log du montant créé (amount_log)")

# 5. Ratio activité (nb transactions / moyenne)
avg_trans = df['nb_transactions_24h'].mean()
df['activity_ratio'] = df['nb_transactions_24h'] / avg_trans
print("\n✅ Ratio d'activité créé (activity_ratio)")

# 6. Score de risque composite
df['risk_score'] = (
    df['changement_localisation'] * 2 +  # Poids fort
    df['appareil_different'] * 1.5 +     # Poids moyen
    (df['hour'] < 6).astype(int) * 1.5 + # Nuit
    (df['hour'] >= 23).astype(int) * 1   # Tard le soir
)
print("\n✅ Score de risque composite créé")

print(f"\n📊 Total features : {len(df.columns)}")
print("="*70)

In [ ]:
# Aperçu des nouvelles features
print("\n👀 Aperçu des nouvelles features :")
new_features = ['day_of_week', 'is_weekend', 'period_of_day', 'amount_category', 
                'amount_log', 'activity_ratio', 'risk_score']
df[['transaction_id', 'is_fraud'] + new_features].head(10)

## 3️⃣ Encodage des variables catégorielles

In [ ]:
print("🔤 ENCODAGE DES VARIABLES CATÉGORIELLES")
print("="*70)

# Variables à encoder
categorical_vars = ['transaction_type', 'city', 'operator', 'period_of_day', 'amount_category']

print(f"\nVariables à encoder : {categorical_vars}")

# One-Hot Encoding
df_encoded = pd.get_dummies(df, columns=categorical_vars, prefix=categorical_vars, drop_first=False)

print(f"\n✅ Encodage terminé")
print(f"Colonnes avant encodage : {len(df.columns)}")
print(f"Colonnes après encodage : {len(df_encoded.columns)}")
print(f"Nouvelles colonnes créées : {len(df_encoded.columns) - len(df.columns)}")
print("="*70)

In [ ]:
# Aperçu des colonnes encodées
print("\n📋 Colonnes après encodage :")
print(df_encoded.columns.tolist())

## 4️⃣ Sélection des features pour le modèle

In [ ]:
print("🎯 SÉLECTION DES FEATURES")
print("="*70)

# Colonnes à exclure
exclude_cols = ['transaction_id', 'date', 'user_id', 'is_fraud', 'fraud_type', 'amount_category']

# Features pour le modèle
feature_cols = [col for col in df_encoded.columns if col not in exclude_cols]

print(f"\nNombre de features sélectionnées : {len(feature_cols)}")
print(f"\nFeatures :")
for i, feature in enumerate(feature_cols, 1):
    print(f"   {i:2d}. {feature}")

print("="*70)

In [ ]:
# Séparation X et y
X = df_encoded[feature_cols]
y = df_encoded['is_fraud']

print("📊 Dimensions :")
print(f"X : {X.shape}")
print(f"y : {y.shape}")
print(f"\nDistribution y :")
print(y.value_counts())

## 5️⃣ Normalisation des features numériques

In [ ]:
print("📐 NORMALISATION DES FEATURES NUMÉRIQUES")
print("="*70)

# Identifier les colonnes numériques à normaliser
numeric_features = ['hour', 'amount', 'nb_transactions_24h', 'day_of_week', 
                   'month', 'day', 'amount_log', 'activity_ratio', 'risk_score']

# Filtrer celles qui existent dans X
numeric_features = [col for col in numeric_features if col in X.columns]

print(f"\nFeatures numériques à normaliser : {len(numeric_features)}")
for feat in numeric_features:
    print(f"   • {feat}")

# Normalisation avec StandardScaler
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[numeric_features] = scaler.fit_transform(X[numeric_features])

print("\n✅ Normalisation terminée (mean=0, std=1)")
print("="*70)

In [ ]:
# Vérification de la normalisation
print("\n🔍 Vérification (moyennes et écarts-types) :")
for feat in numeric_features[:3]:  # Afficher 3 exemples
    mean = X_scaled[feat].mean()
    std = X_scaled[feat].std()
    print(f"   {feat:25s} : mean={mean:.4f}, std={std:.4f}")

## 6️⃣ Division Train / Test

In [ ]:
print("✂️  DIVISION TRAIN / TEST")
print("="*70)

# Split stratifié (important pour classes déséquilibrées)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, 
    test_size=0.2,  # 80% train, 20% test
    random_state=42,
    stratify=y  # Maintient la proportion de fraudes
)

print(f"\n📊 Dimensions :")
print(f"   Train : {X_train.shape}")
print(f"   Test  : {X_test.shape}")

print(f"\n🚨 Distribution des fraudes :")
print(f"\nTrain set :")
print(f"   Normal : {(y_train==0).sum():,} ({(y_train==0).mean()*100:.2f}%)")
print(f"   Fraude : {(y_train==1).sum():,} ({(y_train==1).mean()*100:.2f}%)")

print(f"\nTest set :")
print(f"   Normal : {(y_test==0).sum():,} ({(y_test==0).mean()*100:.2f}%)")
print(f"   Fraude : {(y_test==1).sum():,} ({(y_test==1).mean()*100:.2f}%)")

print("\n✅ Split stratifié réussi (proportions maintenues)")
print("="*70)

## 7️⃣ Gestion du déséquilibre (SMOTE)

**SMOTE** (Synthetic Minority Over-sampling Technique) crée des exemples synthétiques de la classe minoritaire.

In [ ]:
print("⚖️  GESTION DU DÉSÉQUILIBRE AVEC SMOTE")
print("="*70)

print(f"\nAvant SMOTE :")
print(f"   Train set : {len(y_train):,} samples")
print(f"   Normal : {(y_train==0).sum():,}")
print(f"   Fraude : {(y_train==1).sum():,}")
print(f"   Ratio : 1:{(y_train==0).sum()//(y_train==1).sum()}")

# Application de SMOTE (sur le train set uniquement !)
smote = SMOTE(sampling_strategy=0.5, random_state=42)  # 50% de fraudes après SMOTE
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"\nAprès SMOTE :")
print(f"   Train set : {len(y_train_balanced):,} samples")
print(f"   Normal : {(y_train_balanced==0).sum():,}")
print(f"   Fraude : {(y_train_balanced==1).sum():,}")
print(f"   Ratio : 1:{(y_train_balanced==0).sum()//(y_train_balanced==1).sum()}")

print("\n✅ Rééquilibrage réussi")
print("⚠️  Note : SMOTE appliqué UNIQUEMENT sur le train set")
print("="*70)

In [ ]:
# Visualisation avant/après SMOTE
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Avant SMOTE
y_train.value_counts().plot(kind='bar', ax=axes[0], color=['lightgreen', 'salmon'])
axes[0].set_title('Distribution AVANT SMOTE', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Classe')
axes[0].set_ylabel('Nombre de samples')
axes[0].set_xticklabels(['Normal', 'Fraude'], rotation=0)
axes[0].grid(True, alpha=0.3, axis='y')

# Après SMOTE
pd.Series(y_train_balanced).value_counts().plot(kind='bar', ax=axes[1], color=['lightgreen', 'salmon'])
axes[1].set_title('Distribution APRÈS SMOTE', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Classe')
axes[1].set_ylabel('Nombre de samples')
axes[1].set_xticklabels(['Normal', 'Fraude'], rotation=0)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 8️⃣ Sauvegarde des données préparées

In [ ]:
print("💾 SAUVEGARDE DES DONNÉES PRÉPARÉES")
print("="*70)

# Sauvegarder les datasets
import pickle

data_to_save = {
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test,
    'X_train_balanced': X_train_balanced,
    'y_train_balanced': y_train_balanced,
    'scaler': scaler,
    'feature_names': feature_cols
}

with open('../data/preprocessed_data.pkl', 'wb') as f:
    pickle.dump(data_to_save, f)

print("\n✅ Données sauvegardées : ../data/preprocessed_data.pkl")
print("\nContenu :")
for key in data_to_save.keys():
    if key not in ['scaler', 'feature_names']:
        print(f"   • {key:25s} : {data_to_save[key].shape}")
    else:
        print(f"   • {key:25s} : {type(data_to_save[key]).__name__}")

print("="*70)

## 📌 Résumé du preprocessing

### ✅ Étapes réalisées

1. **Feature Engineering** :
   - ✅ Features temporelles (jour semaine, weekend, mois)
   - ✅ Période du jour (nuit, matin, après-midi, soir)
   - ✅ Catégories de montant
   - ✅ Log du montant (normalisation)
   - ✅ Ratio d'activité
   - ✅ Score de risque composite

2. **Encodage** :
   - ✅ One-Hot Encoding des variables catégorielles
   - ✅ 5 variables encodées

3. **Normalisation** :
   - ✅ StandardScaler sur features numériques
   - ✅ Mean=0, Std=1

4. **Split Train/Test** :
   - ✅ 80/20 stratifié
   - ✅ Proportions de fraude maintenues

5. **Gestion du déséquilibre** :
   - ✅ SMOTE appliqué (sampling_strategy=0.5)
   - ✅ Uniquement sur train set
   - ✅ Ratio passé de 1:16 à 1:2

6. **Sauvegarde** :
   - ✅ Données prêtes pour la modélisation
   - ✅ Scaler sauvegardé pour inférence future

### 📊 Statistiques finales

- **Features** : 30+ features engineered
- **Train set** : ~80K samples (avant SMOTE) → ~120K (après SMOTE)
- **Test set** : ~20K samples (inchangé)
- **Ratio fraude** : ~50% train (après SMOTE), ~6% test (réaliste)

### 🚀 Prochaine étape

**Notebook 03** : Modélisation
- Régression Logistique (baseline)
- Random Forest
- XGBoost (optionnel)
- Évaluation complète (Recall, Precision, F1, AUC-ROC)
- Feature Importance
- Optimisation des hyperparamètres